In [9]:
import pandas as pd
import glob
import os
from sqlalchemy import create_engine

In [19]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")
except Exception as error:
    print(f"Failed to connect to air_quality: {error}")

Connected to air_quality!


In [11]:
dataset_folder = "../PRSA_Data_20130301-20170228"

files = glob.glob(
    os.path.join(dataset_folder, "*.csv")
)

print("Datasets found:", len(files))

for file in files:
    print(os.path.basename(file))

Datasets found: 12
PRSA_Data_Aotizhongxin_20130301-20170228.csv
PRSA_Data_Changping_20130301-20170228.csv
PRSA_Data_Dingling_20130301-20170228.csv
PRSA_Data_Dongsi_20130301-20170228.csv
PRSA_Data_Guanyuan_20130301-20170228.csv
PRSA_Data_Gucheng_20130301-20170228.csv
PRSA_Data_Huairou_20130301-20170228.csv
PRSA_Data_Nongzhanguan_20130301-20170228.csv
PRSA_Data_Shunyi_20130301-20170228.csv
PRSA_Data_Tiantan_20130301-20170228.csv
PRSA_Data_Wanliu_20130301-20170228.csv
PRSA_Data_Wanshouxigong_20130301-20170228.csv


In [12]:
file = files[0]

df = pd.read_csv(file)

print("Dataset:", os.path.basename(file))
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Dataset: PRSA_Data_Aotizhongxin_20130301-20170228.csv
Rows: 35064
Columns: 18


,No,year,month,day,hour,PM2.5,PM10,SO2,NO2,CO,O3,TEMP,PRES,DEWP,RAIN,wd,WSPM,station
0,1,2013,3,1,0,4.0,4.0,4.0,7.0,300.0,77.0,-0.7,1023.0,-18.8,0.0,NNW,4.4,Aotizhongxin
1,2,2013,3,1,1,8.0,8.0,4.0,7.0,300.0,77.0,-1.1,1023.2,-18.2,0.0,N,4.7,Aotizhongxin
2,3,2013,3,1,2,7.0,7.0,5.0,10.0,300.0,73.0,-1.1,1023.5,-18.2,0.0,NNW,5.6,Aotizhongxin
3,4,2013,3,1,3,6.0,6.0,11.0,11.0,300.0,72.0,-1.4,1024.5,-19.4,0.0,NW,3.1,Aotizhongxin
4,5,2013,3,1,4,3.0,3.0,12.0,12.0,300.0,72.0,-2.0,1025.2,-19.5,0.0,N,2.0,Aotizhongxin


In [13]:
for file in files:

    # Extract dataset
    df = pd.read_csv(file)

    # Get the station name from the filename
    filename = os.path.basename(file)

    table_name = filename.replace(
        "PRSA_Data_", ""
    ).replace(
        "_20130301-20170228.csv", ""
    ).lower()

    print(f"Loading {table_name}...")

    # Load data into MySQL
    df.to_sql(
        table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=1000
    )

    print(f"{table_name} completed: {len(df):,} rows")

print("\nALL DATASETS LOADED SUCCESSFULLY!")

Loading aotizhongxin...
aotizhongxin completed: 35,064 rows
Loading changping...
changping completed: 35,064 rows
Loading dingling...
dingling completed: 35,064 rows
Loading dongsi...
dongsi completed: 35,064 rows
Loading guanyuan...
guanyuan completed: 35,064 rows
Loading gucheng...
gucheng completed: 35,064 rows
Loading huairou...
huairou completed: 35,064 rows
Loading nongzhanguan...
nongzhanguan completed: 35,064 rows
Loading shunyi...
shunyi completed: 35,064 rows
Loading tiantan...
tiantan completed: 35,064 rows
Loading wanliu...
wanliu completed: 35,064 rows
Loading wanshouxigong...
wanshouxigong completed: 35,064 rows

ALL DATASETS LOADED SUCCESSFULLY!


In [ ]:
table_names = pd.read_sql(
    "SHOW TABLES FROM air_quality",
    engine
)

table_names = table_names.iloc[:, 0].tolist()

counts = []

for table_name in table_names:
    row_count = pd.read_sql(
        f"SELECT COUNT(*) AS total_records FROM `{table_name}`",
        engine
    ).iloc[0, 0]
    counts.append({
        "table_name": table_name,
        "total_records": int(row_count)
    })

result = pd.DataFrame(counts)

result

NameError: name 'TABLE_NAME' is not defined